# Car Engine Failure Predictor
**Author:** Luna Issaias Sbahtu | Arizona State University

Predicts whether a car engine is at risk of failure based on diagnostic parameters using a **Random Forest** classifier. Includes a desktop GUI built in Tkinter.

**Pipeline:**
1. Synthetic data generation with physics-informed risk scoring
2. Random Forest training with balanced class weights
3. Model evaluation — accuracy, precision, recall, confusion matrix
4. Feature importance analysis
5. Desktop GUI for real-time prediction (`python predictor.py`)

## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import joblib
import warnings
warnings.filterwarnings('ignore')
print('Libraries loaded.')

## 2. Data Generation
Generates 5,000 synthetic engine telemetry samples with realistic distributions and physics-informed failure probability.

In [ ]:
np.random.seed(42)
n = 5000

mileage                 = np.random.uniform(10000, 300000, n)
engine_temp             = np.clip(np.random.normal(90, 10, n), 60, 130)
oil_pressure            = np.clip(np.random.normal(3.0, 0.7, n), 0.5, 6.0)
rpm                     = np.random.uniform(600, 4500, n)
error_code_count        = np.clip(np.random.poisson(1.0, n), 0, 10)
time_since_last_service = np.random.uniform(0, 700, n)
avg_trip_duration       = np.random.uniform(5, 90, n)
city_drive_ratio        = np.random.uniform(0, 1, n)

risk = ((mileage - 100000)/100000 + (engine_temp - 90)/20 +
        (3.0 - oil_pressure)/1.5 + error_code_count*0.5 +
        (time_since_last_service - 365)/365 + city_drive_ratio*0.5)
prob_fail = 1 / (1 + np.exp(-(risk - 0.5)))
will_fail = np.random.binomial(1, prob_fail)

df = pd.DataFrame({
    'mileage': mileage, 'engine_temp': engine_temp,
    'oil_pressure': oil_pressure, 'rpm': rpm,
    'error_code_count': error_code_count,
    'time_since_last_service': time_since_last_service,
    'avg_trip_duration': avg_trip_duration,
    'city_drive_ratio': city_drive_ratio,
    'will_fail_soon': will_fail
})

print(f'Dataset shape: {df.shape}')
print(f'Failure rate: {will_fail.mean():.2%}')
df.describe().round(2)

## 3. Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
features = ['mileage', 'engine_temp', 'oil_pressure', 'rpm',
            'error_code_count', 'time_since_last_service', 'avg_trip_duration', 'city_drive_ratio']
for ax, feat in zip(axes.flatten(), features):
    for label, color in [(0, 'steelblue'), (1, 'tomato')]:
        subset = df[df['will_fail_soon'] == label][feat]
        subset.plot.kde(ax=ax, label='No Failure' if label == 0 else 'Failure', color=color)
    ax.set_title(feat.replace('_', ' ').title(), fontsize=10)
    ax.legend(fontsize=8)
plt.suptitle('Feature Distributions by Failure Label', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Model Training — Random Forest

In [ ]:
X = df.drop('will_fail_soon', axis=1)
y = df['will_fail_soon']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

model = RandomForestClassifier(
    n_estimators=300, random_state=42, n_jobs=-1, class_weight='balanced')
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print('=== Classification Report ===')
print(classification_report(y_test, y_pred, target_names=['No Failure', 'Failure']))

## 5. Confusion Matrix & Feature Importance

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ConfusionMatrixDisplay(confusion_matrix(y_test, y_pred),
                       display_labels=['No Failure', 'Failure']).plot(ax=axes[0], colorbar=False)
axes[0].set_title('Confusion Matrix', fontweight='bold')

importances = pd.Series(model.feature_importances_, index=X.columns).sort_values()
importances.plot.barh(ax=axes[1], color='steelblue')
axes[1].set_title('Feature Importance', fontweight='bold')
axes[1].set_xlabel('Importance Score')

plt.tight_layout()
plt.show()

## 6. Sample Prediction & Save Model

In [ ]:
joblib.dump(model, 'engine_failure_predictor.joblib')
print('Model saved to engine_failure_predictor.joblib')

sample = pd.DataFrame([{
    'mileage': 180000, 'engine_temp': 105, 'oil_pressure': 2.2,
    'rpm': 3200, 'error_code_count': 3, 'time_since_last_service': 450,
    'avg_trip_duration': 25, 'city_drive_ratio': 0.8
}])
prob = model.predict_proba(sample)[0][1]
status = 'RISK' if prob >= 0.7 else 'OK'
print(f'Sample prediction -> Failure probability: {prob:.2%} | Status: {status}')
print()
print('To launch the desktop GUI, run:  python predictor.py')